# 🛰️ Landslide Detection from Sentinel-2 Imagery
## Deep Learning with PyTorch Lightning

---

**Author:** Remote Sensing ML Research  
**Framework:** PyTorch + PyTorch Lightning  
**Dataset:** Sentinel-2 image patches (ESRI Classified Tiles format)  
**Task:** Binary image classification (Landslide / Non-landslide)

### Notebook Sections
1. 📦 Install Dependencies
2. 📂 Dataset Setup & Exploration
3. 🏗️ Model & Training Setup
4. 🚀 Train ResNet18 with Focal Loss *(recommended)*
5. 🔬 Train Baseline CNN for Comparison
6. 📊 Evaluation & Metrics
7. 🎨 Visualizations (ROC, Confusion Matrix, Loss Curves)
8. 🔍 Grad-CAM Interpretability
9. 📈 Model Comparison Table
10. 🤖 Inference Demo

## 1. 📦 Install Dependencies

Run this cell once per Colab session.

In [ ]:
# Install all required packages
!pip install -q torch torchvision pytorch-lightning
!pip install -q rasterio scikit-learn pandas numpy scipy
!pip install -q torchmetrics grad-cam seaborn matplotlib
!pip install -q tensorboard PyYAML tqdm rich

print('✅ All dependencies installed successfully!')

## 2. 📂 Dataset Setup & Exploration

Upload your dataset directory or mount Google Drive.

In [ ]:
import os
import sys
from pathlib import Path

# ─── Option A: Mount Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Update this path to where your LandslideData folder is on Drive:
DATA_ROOT = '/content/drive/MyDrive/LandslideData'

# ─── Option B: Upload as ZIP ─────────────────────────────────────────
# from google.colab import files
# import zipfile
# uploaded = files.upload()  # Upload LandslideData.zip
# with zipfile.ZipFile('LandslideData.zip', 'r') as z:
#     z.extractall('/content/')
# DATA_ROOT = '/content/LandslideData'

# ─── Clone project repo (if using GitHub) ───────────────────────────
# !git clone https://github.com/yourusername/landslide-detection.git /content/project
# sys.path.insert(0, '/content/project')

# ─── Or copy src/ from Drive ─────────────────────────────────────────
PROJECT_SRC = '/content/drive/MyDrive/landslide_dl_project'
sys.path.insert(0, PROJECT_SRC)

print(f'DATA_ROOT = {DATA_ROOT}')
print(f'Exists: {Path(DATA_ROOT).exists()}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from pathlib import Path

from src.dataset import build_dataframe, CLASS_NAMES

# ── Build dataset index ──────────────────────────────────────────────
df = build_dataframe(DATA_ROOT)
print(f'Total patches : {len(df)}')
print(f'Class distribution:\n{df["class_name"].value_counts()}')
print(f'\nClass balance: {df["class_idx"].mean():.2%} landslide')
print('\nSample rows:')
df.head()

In [ ]:
from src.dataset import LandslideDataset

# Build a dataset without transforms for visualization
full_ds = LandslideDataset(df, transform=None, normalize=False)

# Pick samples from each class
landslide_idx = df[df['class_idx'] == 1].index[:4].tolist()
nonls_idx     = df[df['class_idx'] == 0].index[:4].tolist()
sample_indices = nonls_idx + landslide_idx

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, idx in enumerate(sample_indices):
    rgb = full_ds.get_rgb_image(idx)
    axes[i].imshow(rgb)
    label = df.iloc[idx]['class_name']
    axes[i].set_title(f'{label}', fontsize=12,
                      color='red' if label == 'Landslide' else 'green',
                      fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Patches: Row 1 = Non-Landslide | Row 2 = Landslide',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/sample_patches.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample patches saved.')

In [ ]:
# ── Band statistics per class ────────────────────────────────────────
BAND_NAMES = ['Red', 'Green', 'Blue', 'NIR']
class_means = {cls: {b: [] for b in BAND_NAMES} for cls in CLASS_NAMES}

for _, row in df.iterrows():
    with rasterio.open(row['image_path']) as src:
        for j, bname in enumerate(BAND_NAMES):
            class_means[row['class_name']][bname].append(
                src.read(j + 1).mean()
            )

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for j, bname in enumerate(BAND_NAMES):
    vals = [class_means[c][bname] for c in CLASS_NAMES]
    axes[j].boxplot(vals, labels=CLASS_NAMES, patch_artist=True,
                    boxprops=dict(facecolor='#457B9D', alpha=0.7))
    axes[j].set_title(f'{bname} Band', fontweight='bold')
    axes[j].set_ylabel('Mean DN Value')
    axes[j].grid(axis='y', alpha=0.3)

plt.suptitle('Band-wise DN Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/band_distributions.png', dpi=150)
plt.show()

## 3. 🏗️ Model & Training Setup

Configure shared hyperparameters used for all experiments.

In [ ]:
from src.utils import set_seed

# ── Global configuration ─────────────────────────────────────────────
CONFIG = {
    'data_root'   : DATA_ROOT,
    'num_bands'   : 4,
    'batch_size'  : 8,
    'num_workers' : 2,        # Colab has 2 CPUs
    'val_split'   : 0.2,
    'seed'        : 42,
    'max_epochs'  : 60,
    'lr'          : 1e-3,
    'weight_decay': 1e-4,
    'patience'    : 12,
    'output_dir'  : '/content/outputs',
}

set_seed(CONFIG['seed'])
import os; os.makedirs(CONFIG['output_dir'], exist_ok=True)

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:<20} = {v}')

In [ ]:
from src.datamodule import LandslideDataModule

# Shared DataModule (reused across experiments)
datamodule = LandslideDataModule(
    data_root    = CONFIG['data_root'],
    batch_size   = CONFIG['batch_size'],
    num_workers  = CONFIG['num_workers'],
    val_split    = CONFIG['val_split'],
    num_bands    = CONFIG['num_bands'],
    seed         = CONFIG['seed'],
    pin_memory   = True,
)
datamodule.setup()

print(f'Train size: {len(datamodule.train_dataset)}')
print(f'Val size  : {len(datamodule.val_dataset)}')
print(f'Class weights: {datamodule.class_weights}')

# Verify a batch
import torch
batch = next(iter(datamodule.train_dataloader()))
images, labels = batch
print(f'\nBatch: images {images.shape} | labels {labels.shape}')
print(f'Image dtype: {images.dtype}, range: [{images.min():.2f}, {images.max():.2f}]')

## 4. 🚀 Train ResNet18 with Focal Loss *(Recommended)*

In [ ]:
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import (
    EarlyStopping, ModelCheckpoint, LearningRateMonitor, RichProgressBar
)
from pytorch_lightning.loggers import TensorBoardLogger

from src.model import get_model
from src.losses import get_loss_fn
from src.lightning_module import LandslideClassifier

# ── Build model + loss ───────────────────────────────────────────────
model_r18_focal = get_model('resnet18', num_bands=4, pretrained=True, freeze_backbone=True)
loss_focal      = get_loss_fn('focal', class_weights=datamodule.class_weights, focal_gamma=2.0)

module_r18_focal = LandslideClassifier(
    model          = model_r18_focal,
    loss_fn        = loss_focal,
    lr             = CONFIG['lr'],
    optimizer_name = 'adamw',
    scheduler_name = 'cosine',
    weight_decay   = CONFIG['weight_decay'],
    max_epochs     = CONFIG['max_epochs'],
    unfreeze_epoch = 10,
)

# ── Callbacks ────────────────────────────────────────────────────────
ckpt_r18_focal = ModelCheckpoint(
    dirpath  = f"{CONFIG['output_dir']}/resnet18_focal/checkpoints",
    filename = 'resnet18_focal_best',
    monitor  = 'val_auroc',
    mode     = 'max',
    save_top_k = 1,
    verbose  = True,
)
early_stop = EarlyStopping(monitor='val_auroc', mode='max', patience=CONFIG['patience'])
lr_monitor = LearningRateMonitor(logging_interval='epoch')

tb_logger = TensorBoardLogger(
    save_dir = f"{CONFIG['output_dir']}/resnet18_focal/logs",
    name     = 'resnet18_focal',
)

trainer_r18_focal = Trainer(
    max_epochs        = CONFIG['max_epochs'],
    accelerator       = 'auto',
    devices           = 1,
    callbacks         = [ckpt_r18_focal, early_stop, lr_monitor, RichProgressBar()],
    logger            = tb_logger,
    log_every_n_steps = 1,
    deterministic     = True,
)

trainer_r18_focal.fit(module_r18_focal, datamodule=datamodule)
print(f'\nBest checkpoint: {ckpt_r18_focal.best_model_path}')
print(f'Best val AUROC : {ckpt_r18_focal.best_model_score:.4f}')

## 5. 🔬 Train Baseline CNN for Comparison

In [ ]:
from src.model import get_model
from src.losses import get_loss_fn
from src.lightning_module import LandslideClassifier

# Reset data seed for fair comparison
set_seed(CONFIG['seed'])
datamodule.setup()   # re-run to get same split

# ── Baseline CNN + Focal Loss ────────────────────────────────────────
model_baseline = get_model('baseline_cnn', num_bands=4)
loss_focal_b   = get_loss_fn('focal', class_weights=datamodule.class_weights, focal_gamma=2.0)

module_baseline = LandslideClassifier(
    model          = model_baseline,
    loss_fn        = loss_focal_b,
    lr             = CONFIG['lr'],
    optimizer_name = 'adamw',
    scheduler_name = 'cosine',
    weight_decay   = CONFIG['weight_decay'],
    max_epochs     = CONFIG['max_epochs'],
    unfreeze_epoch = None,   # nothing to unfreeze in baseline
)

ckpt_baseline = ModelCheckpoint(
    dirpath  = f"{CONFIG['output_dir']}/baseline_focal/checkpoints",
    filename = 'baseline_focal_best',
    monitor  = 'val_auroc',
    mode     = 'max',
    save_top_k = 1,
)
early_stop_b = EarlyStopping(monitor='val_auroc', mode='max', patience=CONFIG['patience'])

tb_logger_b = TensorBoardLogger(
    save_dir = f"{CONFIG['output_dir']}/baseline_focal/logs",
    name     = 'baseline_focal',
)

trainer_baseline = Trainer(
    max_epochs        = CONFIG['max_epochs'],
    accelerator       = 'auto',
    devices           = 1,
    callbacks         = [ckpt_baseline, early_stop_b,
                         LearningRateMonitor(logging_interval='epoch'),
                         RichProgressBar()],
    logger            = tb_logger_b,
    log_every_n_steps = 1,
    deterministic     = True,
)

trainer_baseline.fit(module_baseline, datamodule=datamodule)
print(f'\nBest checkpoint: {ckpt_baseline.best_model_path}')
print(f'Best val AUROC : {ckpt_baseline.best_model_score:.4f}')

## 6. 📊 Evaluation & Metrics

Load best checkpoints and run full evaluation with threshold optimization.

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
from src.utils import threshold_optimization, save_metrics_csv

def evaluate_model(module, datamodule, model_tag, output_dir):
    """Full evaluation: metrics + threshold opt + reports."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    module.to(device)
    module.eval()

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in datamodule.val_dataloader():
            logits = module(images.to(device))
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())

    y_prob  = np.concatenate(all_probs)
    y_true  = np.concatenate(all_labels)

    # Threshold optimization
    best_thr, best_f1 = threshold_optimization(y_true, y_prob, metric='f1')
    y_pred = (y_prob >= best_thr).astype(int)

    metrics = {
        'accuracy'       : round(accuracy_score(y_true, y_pred), 4),
        'precision'      : round(precision_score(y_true, y_pred, zero_division=0), 4),
        'recall'         : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'f1'             : round(f1_score(y_true, y_pred, zero_division=0), 4),
        'auc'            : round(roc_auc_score(y_true, y_prob), 4),
        'best_threshold' : round(best_thr, 4),
    }

    print(f'\n═══ {model_tag.upper()} ═══')
    for k, v in metrics.items():
        print(f'  {k:<20} : {v}')
    print('\n' + classification_report(y_true, y_pred,
          target_names=['Non-Landslide', 'Landslide']))

    # Save CSV
    csv_path = f"{output_dir}/metrics_summary.csv"
    save_metrics_csv(metrics, csv_path, model_name=model_tag)

    return metrics, y_true, y_prob, y_pred

In [ ]:
# ── Load best checkpoints ────────────────────────────────────────────
from src.lightning_module import LandslideClassifier

best_r18 = LandslideClassifier.load_from_checkpoint(
    ckpt_r18_focal.best_model_path,
    model   = get_model('resnet18', num_bands=4, pretrained=False),
    loss_fn = get_loss_fn('focal', class_weights=datamodule.class_weights),
    strict  = False,
)

best_base = LandslideClassifier.load_from_checkpoint(
    ckpt_baseline.best_model_path,
    model   = get_model('baseline_cnn', num_bands=4),
    loss_fn = get_loss_fn('focal', class_weights=datamodule.class_weights),
    strict  = False,
)

# ── Evaluate both ────────────────────────────────────────────────────
metrics_r18,  y_true, y_prob_r18,  y_pred_r18  = evaluate_model(
    best_r18,   datamodule, 'resnet18_focal',  CONFIG['output_dir']
)
metrics_base, _,      y_prob_base, y_pred_base = evaluate_model(
    best_base,  datamodule, 'baseline_focal', CONFIG['output_dir']
)

## 7. 📊 Visualizations

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model_tag, y_prob in [
    (axes[0], 'ResNet18 (Focal)', y_prob_r18),
    (axes[1], 'Baseline CNN (Focal)', y_prob_base),
]:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2.5, color='#E63946', label=f'AUC = {roc_auc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random')
    ax.fill_between(fpr, tpr, alpha=0.08, color='#E63946')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curve – {model_tag}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/plots/roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from src.utils import plot_confusion_matrix
import os
os.makedirs(f"{CONFIG['output_dir']}/plots", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model_tag, y_pred in [
    (axes[0], 'ResNet18 (Focal)', y_pred_r18),
    (axes[1], 'Baseline CNN (Focal)', y_pred_base),
]:
    import seaborn as sns
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-LS', 'Landslide'],
                yticklabels=['Non-LS', 'Landslide'],
                ax=ax, annot_kws={'size': 14})
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'Confusion Matrix – {model_tag}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/plots/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# ── Extract and plot training curves from TensorBoard logs ───────────
from src.utils import extract_tb_scalars, plot_loss_curves

for model_tag, log_dir in [
    ('resnet18_focal',  f"{CONFIG['output_dir']}/resnet18_focal/logs/resnet18_focal/v1"),
    ('baseline_focal', f"{CONFIG['output_dir']}/baseline_focal/logs/baseline_focal/v1"),
]:
    scalars = extract_tb_scalars(log_dir)
    if 'train_loss_epoch' in scalars and 'val_loss' in scalars:
        train_losses = scalars['train_loss_epoch']['value'].tolist()
        val_losses   = scalars['val_loss']['value'].tolist()
        min_len = min(len(train_losses), len(val_losses))
        plot_loss_curves(
            train_losses[:min_len], val_losses[:min_len],
            out_path=f"{CONFIG['output_dir']}/plots/loss_curves_{model_tag}.png",
            title=f'Loss Curves – {model_tag.replace("_", " ").title()}',
        )
        plt.show()
    else:
        print(f'Available scalars for {model_tag}: {list(scalars.keys())}')

## 8. 🔍 Grad-CAM Interpretability

Visualize which spatial regions the ResNet18 model focuses on when detecting landslides.

In [ ]:
from src.inference import LandslideInferencer
from pathlib import Path

# ── Initialize inferencer from best checkpoint ───────────────────────
inferencer = LandslideInferencer(
    ckpt_path = ckpt_r18_focal.best_model_path,
    device    = 'auto',
    threshold = metrics_r18['best_threshold'],
    num_bands = 4,
)

# ── Pick a few landslide examples ───────────────────────────────────
landslide_samples = df[df['class_idx'] == 1]['image_path'].head(4).tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i, img_path in enumerate(landslide_samples):
    # Original RGB
    import rasterio
    with rasterio.open(img_path) as src:
        rgb = src.read([1, 2, 3]).astype(np.float32)
    p2, p98 = np.percentile(rgb, [2, 98])
    rgb_vis = np.clip((rgb - p2) / (p98 - p2 + 1e-8), 0, 1).transpose(1, 2, 0)

    # Grad-CAM
    try:
        result, overlay = inferencer.predict_with_gradcam(img_path)
        axes[0, i].imshow(rgb_vis)
        axes[0, i].set_title(f'Original\nP(LS)={result["probability"]:.2f}', fontsize=10)
        axes[0, i].axis('off')

        axes[1, i].imshow(overlay)
        axes[1, i].set_title(f'Grad-CAM Overlay', fontsize=10, color='red')
        axes[1, i].axis('off')
    except Exception as e:
        print(f'Grad-CAM failed for {img_path}: {e}')

plt.suptitle('Grad-CAM: ResNet18 Attention on Landslide Patches',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/plots/gradcam_samples.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. 📈 Model Comparison Table

Summary of all experiments including loss function comparison.

In [ ]:
import pandas as pd

# Load the metrics CSV we've been appending to
csv_path = f"{CONFIG['output_dir']}/metrics_summary.csv"
metrics_df = pd.read_csv(csv_path)

# Display nicely
print('Model Performance Comparison')
print('=' * 70)
display_df = metrics_df.set_index('model')
print(display_df.to_string())

# Fancy table with styling
display_df.style.background_gradient(cmap='RdYlGn', subset=['accuracy','f1','auc','recall'])

In [ ]:
from src.utils import plot_metrics_bar

# Prepare for grouped bar chart
bar_df = metrics_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'auc']].rename(
    columns={'model': 'Model', 'accuracy': 'Accuracy', 'precision': 'Precision',
             'recall': 'Recall', 'f1': 'F1-Score', 'auc': 'AUC'}
)

plot_metrics_bar(
    bar_df,
    out_path=f"{CONFIG['output_dir']}/plots/model_comparison.png",
    title='Performance Comparison: ResNet18 vs Baseline CNN',
)
plt.show()

## 10. 🤖 Inference Demo

Demonstrate single image prediction, batch prediction, and probability interpretation.

In [ ]:
from src.inference import load_inferencer_from_ckpt

# ── Load best ResNet18 model ──────────────────────────────────────────
inf = load_inferencer_from_ckpt(
    ckpt_path = ckpt_r18_focal.best_model_path,
    device    = 'auto',
    threshold = metrics_r18['best_threshold'],  # optimized threshold
    num_bands = 4,
)

# ── Single image prediction ───────────────────────────────────────────
sample_image = df.iloc[0]['image_path']
result = inf.predict_single(sample_image)

print('Single Image Prediction:')
print(f'  Image       : {result["image_path"]}')
print(f'  P(Landslide): {result["probability"]:.4f} = {result["probability"]*100:.1f}%')
print(f'  Predicted   : {result["class_name"]} (idx={result["class_idx"]})')
print(f'  Threshold   : {result["threshold"]}')

In [ ]:
# ── Batch prediction on entire images directory ───────────────────────
images_dir = Path(DATA_ROOT) / 'images'
batch_results = inf.predict_batch(images_dir, threshold=metrics_r18['best_threshold'])

print(f'Batch inference on {len(batch_results)} images:')
print(batch_results['class_name'].value_counts())
print('\nTop predictions by probability:')
print(batch_results.sort_values('probability', ascending=False).head(10).to_string(index=False))

# Save batch results
batch_results.to_csv(f"{CONFIG['output_dir']}/batch_predictions.csv", index=False)
print(f"\nBatch results saved.")

In [ ]:
# ── Demonstrate threshold effect ──────────────────────────────────────
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
print('Threshold sensitivity for single patch:')
print(f'{"Threshold":<12} {"P(LS)":<10} {"Decision"}')
print('-' * 35)
for t in thresholds:
    r = inf.predict_single(sample_image, threshold=t)
    decision = '⚠️  LANDSLIDE' if r['class_idx'] == 1 else '✅ Safe'
    print(f'{t:<12} {r["probability"]:<10.4f} {decision}')

In [ ]:
# ── Summary of saved outputs ──────────────────────────────────────────
import glob

print('📁 Saved outputs:')
for f in sorted(glob.glob(f"{CONFIG['output_dir']}/**/*", recursive=True)):
    if os.path.isfile(f):
        size_kb = os.path.getsize(f) / 1024
        print(f'  {f.replace(CONFIG["output_dir"], "outputs")}  ({size_kb:.1f} KB)')

# Download all outputs as ZIP (optional)
# import shutil
# shutil.make_archive('/content/landslide_outputs', 'zip', CONFIG['output_dir'])
# from google.colab import files
# files.download('/content/landslide_outputs.zip')